### Libraries

In [19]:
import argparse
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# Try to import LightGBM; gracefully fall back if it isn't installed.
try:
    import lightgbm
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


RANDOM_STATE = 42
TARGET_COL = "Heart_Disease"

# Columns that are pure identifiers / free text and carry no predictive signal.
COLUMNS_TO_DROP = ["Column1", "Name", "Last_Visit_Date", "Notes"]

### Load Data

In [20]:
df = pd.read_excel(r"F:\Download\Version1 (Cleaned) (2).xlsx")    
df.shape

(10000, 17)

In [21]:
df.head()

,Column1,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Heart_Disease
0,ID-00001,Sarah Lopez XFT,49,Female,Boston,22.0,120/80,83,190,No,Yes,No Medication,2021-04-15,30 days,A00,Patient feeling fine,0
1,ID-00002,Emily Sanchez HVJ,49,Female,New York,25.8,120/80,80,180,No,No,No Medication,2021-07-10,14 days,No Diagnosis,"BP high, needs recheck",0
2,ID-00003,Anthony Smith GGB,49,Male,New York,22.0,120/80,80,190,No,Former,Aspirin/Metformin,2021-04-06,30 days,B20,No Notes,0
3,ID-00004,Sharon Anderson OBS,80,Female,New York,33.4,120/80,80,250,Yes,Former,No Medication,2024-11-13,30 days,No Diagnosis,Follow up soon,0
4,ID-00005,Kevin Lee IHA,49,Male,Los Angeles,30.1,120/80,80,180,No,No,No Medication,2025-09-27,14 days,I10,Patient feeling fine,1


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Column1            10000 non-null  object 
 1   Name               10000 non-null  object 
 2   Age                10000 non-null  int64  
 3   Gender             10000 non-null  object 
 4   City               10000 non-null  object 
 5   BMI                10000 non-null  float64
 6   Blood_Pressure     10000 non-null  object 
 7   Heart_Rate         10000 non-null  int64  
 8   Cholesterol_Level  10000 non-null  int64  
 9   Diabetic           10000 non-null  object 
 10  Smoker             10000 non-null  object 
 11  Medications        10000 non-null  object 
 12  Last_Visit_Date    10000 non-null  object 
 13  Follow_Up          10000 non-null  object 
 14  Diagnosis_Code     10000 non-null  object 
 15  Notes              10000 non-null  object 
 16  Heart_Disease      1000

In [23]:
df.describe()

,Age,BMI,Heart_Rate,Cholesterol_Level,Heart_Disease
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000
mean,41.808800,22.986000,81.37290,211.442000,0.246100
std,19.152387,3.090008,10.71057,31.087574,0.430759
min,0.000000,15.000000,50.00000,180.000000,0.000000
25%,20.000000,22.000000,80.00000,190.000000,0.000000
50%,49.000000,22.500000,80.00000,190.000000,0.000000
75%,49.000000,22.500000,80.00000,250.000000,0.000000
max,99.000000,35.000000,120.00000,250.000000,1.000000


### Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
 
    # --- Split Blood_Pressure into Systolic / Diastolic -------------------
    if "Blood_Pressure" in df.columns:
        bp_split = df["Blood_Pressure"].astype(str).str.split("/", n=1, expand=True)
        df["Systolic_BP"] = pd.to_numeric(bp_split[0], errors="coerce")
        df["Diastolic_BP"] = pd.to_numeric(bp_split[1], errors="coerce")
        df.drop(columns=["Blood_Pressure"], inplace=True)
 
    # --- Extract numeric days from Follow_Up -------------------------------
    if "Follow_Up" in df.columns:
        df["Follow_Up_Days"] = (
            df["Follow_Up"].astype(str).str.extract(r"(\d+)").astype(float)
        )
        df.drop(columns=["Follow_Up"], inplace=True)
 
    # --- Drop identifiers / free text that don't help prediction -----------
    drop_cols = [c for c in COLUMNS_TO_DROP if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)
 
    print(f"[INFO] Shape after feature engineering: {df.shape}")
    print(f"[INFO] Dropped columns: {drop_cols}")
    return df
 
 
def get_feature_lists(df: pd.DataFrame, target_col: str):
    """Split remaining columns into numeric vs. categorical feature lists."""
    feature_df = df.drop(columns=[target_col])
    numeric_features = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    return numeric_features, categorical_features

### Data Preprocessing

In [ ]:
def build_preprocessor(numeric_features, categorical_features) -> ColumnTransformer:

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
 
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
 
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])
    return preprocessor

### Building Models

In [ ]:
def get_models() -> dict:

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }
 
    if LIGHTGBM_AVAILABLE:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbosity=-1,
        )
    else:
        print(
            "[WARN] `lightgbm` not installed — falling back to "
            "HistGradientBoostingClassifier for the gradient-boosting model. "
            "Run `pip install lightgbm` to use LightGBM instead."
        )
        models["GradientBoosting (HGB)"] = HistGradientBoostingClassifier(
            max_iter=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
 
    return models

### Evaluation & Prediction

In [27]:
def evaluate_model(name, pipeline, X_test, y_test) -> dict:
    """Compute accuracy, ROC-AUC, classification report, and confusion matrix."""
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
 
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, digits=3)
    cm = confusion_matrix(y_test, y_pred)
 
    print(f"\n{'=' * 70}")
    print(f"MODEL: {name}")
    print(f"{'=' * 70}")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print("\nClassification Report:")
    print(report)
    print("Confusion Matrix:")
    print("                 Predicted 0    Predicted 1")
    print(f"Actual 0         {cm[0][0]:<14} {cm[0][1]}")
    print(f"Actual 1         {cm[1][0]:<14} {cm[1][1]}")
 
    return {"Model": name, "Accuracy": acc, "ROC_AUC": auc}

### Full pipeline

In [ ]:
def main(data_path: str):
    if TARGET_COL not in df.columns:
        sys.exit(f"[ERROR] Target column '{TARGET_COL}' not found in dataset.")
 
    df = engineer_features(df)
 
    # Drop any rows where the target itself is missing (can't train/eval on them)
    df = df.dropna(subset=[TARGET_COL])
 
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL].astype(int)
 
    numeric_features, categorical_features = get_feature_lists(df, TARGET_COL)
    print(f"[INFO] Numeric features     ({len(numeric_features)}): {numeric_features}")
    print(f"[INFO] Categorical features ({len(categorical_features)}): {categorical_features}")
 
    # ---- Train/test split (stratified to preserve class ratio) ------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    print(f"\n[INFO] Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
    print(f"[INFO] Class distribution (train): {y_train.value_counts(normalize=True).to_dict()}")
 
    # ---- Build preprocessing pipeline --------------------------------------
    preprocessor = build_preprocessor(numeric_features, categorical_features)
 
    # ---- Train & evaluate every model --------------------------------------
    models = get_models()
    results = []
    fitted_pipelines = {}
 
    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
        pipeline.fit(X_train, y_train)
        fitted_pipelines[name] = pipeline
        results.append(evaluate_model(name, pipeline, X_test, y_test))
 
    # ---- Side-by-side comparison table --------------------------------------
    comparison_df = pd.DataFrame(results).sort_values(by="ROC_AUC", ascending=False)
    comparison_df = comparison_df.reset_index(drop=True)
    comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)
    comparison_df["ROC_AUC"] = comparison_df["ROC_AUC"].round(4)
 
    print(f"\n{'#' * 70}")
    print("MODEL COMPARISON SUMMARY (sorted by ROC-AUC, best first)")
    print(f"{'#' * 70}")
    print(comparison_df.to_string(index=False))
 
    best_model_name = comparison_df.iloc[0]["Model"]
    print(f"\n[RESULT] Best performing model by ROC-AUC: {best_model_name}")
 
    return comparison_df, fitted_pipelines